# ColabNAS


# Các bước chuẩn bị

### Import thư viện

In [1]:
# disable warnings for cleaner console output
import absl.logging
import warnings
absl.logging.set_verbosity(absl.logging.ERROR)
warnings.filterwarnings("ignore")

In [2]:
!pip install --quiet wget
import os
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
import numpy as np

import subprocess
import re
import datetime
import shutil
import glob
import wget
import ssl
import gdown

  Preparing metadata (setup.py) ... done


2026-07-23 14:38:43.086332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784817523.341535      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784817523.426555      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784817524.107730      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784817524.107772      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784817524.107774      57 computation_placer.cc:177] computation placer alr

### Cấp quyền thực thi cho `stm32tflm`

In [3]:
!cp /kaggle/input/datasets/karosvn/stm32tflm-linux/stm32tflm /kaggle/working
!chmod +x /kaggle/working/stm32tflm
!test -x /kaggle/working/stm32tflm && echo "Executable" || echo "Not executable"

Executable


# Cài đặt ColabNAS

In [4]:
FIXED_SEED = 11

In [5]:
# [Fix Randomness] Set global seed for reproducible results across Python, NumPy, and Keras
#tf.keras.utils.set_random_seed(FIXED_SEED)
# [Fix Randomness] Enable deterministic operations to prevent GPU floating-point variations
#tf.config.experimental.enable_op_determinism()

In [6]:
# # Thesis vers
# class ColabNAS :
#     def __init__(self, max_RAM, max_Flash, max_MACC, path_to_training_set, 
#                  val_split, cache=False, input_shape=(50,50,3), save_path='.',
#                  path_to_stm32tflm='/kaggle/working/stm32tflm'):        
#         self.learning_rate = 1e-3
#         self.batch_size = 128
#         self.epochs = 100 

#         self.max_MACC = max_MACC
#         self.max_Flash = max_Flash
#         self.max_RAM = max_RAM
#         self.path_to_training_set = path_to_training_set
#         self.num_classes = len(next(os.walk(path_to_training_set))[1])
#         self.val_split = val_split
#         self.cache = cache
#         self.input_shape = input_shape
#         self.save_path = Path(save_path)

#         self.path_to_trained_models = self.save_path / "trained_models"
#         self.path_to_trained_models.mkdir(parents=True, exist_ok=True)

#         self.path_to_stm32tflm = Path(path_to_stm32tflm)

#         self.load_training_set()

#     # k: number of kernels of the first convolutional layer
#     # c: number of cells added upon the first convolutional layer
#     # pre-processing pipeline not included in MACC computation
#     def Model(self, k, c):
#         # [Fix Randomness] Fix random seed for weight initialization to guarantee consistent model convergence
#         #init = tf.keras.initializers.GlorotUniform(seed=FIXED_SEED)
#         kernel_size = (3, 3)
#         pool_size = (2, 2)
#         pool_strides = (2, 2)

#         number_of_cells_limited = False
#         number_of_mac = 0

#         # inputs = (50, 50, 3) by default
#         inputs = keras.Input(shape=self.input_shape)

#         # preprocessing pipeline
#         x = tf.keras.layers.RandomFlip('horizontal')(inputs)
#         x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
#         x = tf.keras.layers.Rescaling(1./255)(x)
#         x = tf.keras.layers.BatchNormalization()(x)

#         # convolutional base
#         n = k
#         multiplier = 2

#         # first convolutional layer
#         # c_in = 3 now
#         c_in = self.input_shape[2]
#         # x.shape() = (batch_size, 50, 50, n)
#         x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
#         # MAC = 3 * (3x3) * (50x50) * n
#         number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

#         # adding cells
#         for i in range(1, c + 1):
#             if x.shape[1] <= 1 or x.shape[2] <= 1:
#                 number_of_cells_limited = True
#                 break
#             n = int(np.ceil(n * multiplier))
#             multiplier = multiplier - 2**-i
#             # x.shape() = (batch_size, h_old /2, w_old /2, n_old)
#             x = keras.layers.MaxPooling2D(pool_size=pool_size, strides=pool_strides, padding='valid')(x)
#             # c_in = n_old
#             c_in = x.shape[3]
#             # x.shape() = (batch_size, h, w, n_new)
#             x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
#             # MAC = c_in * (3x3) * (hxw) * n_new
#             number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

#         # classifier
#         # x.shape() = (batch_size, n_last)
#         x = keras.layers.GlobalAveragePooling2D()(x)
#         # input_shape = n_last
#         input_shape = x.shape[1]
#         # x.shape() = (batch_size, n_last)
#         x = keras.layers.Dense(n, activation='relu')(x)
#         number_of_mac += (input_shape * x.shape[1])
#         # outputs.shape() = (batch_size, num_classes)
#         outputs = keras.layers.Dense(self.num_classes, activation='softmax')(x)
#         number_of_mac += (x.shape[1] * outputs.shape[1])

#         model = keras.Model(inputs=inputs, outputs=outputs)

#         optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
#         model.compile(optimizer=optimizer,
#                 loss='categorical_crossentropy',
#                 metrics=['accuracy'])
        
#         model.summary()

#         return model, number_of_mac, number_of_cells_limited

#     def load_training_set(self):
#         if 3 == self.input_shape[2]:
#             color_mode = 'rgb'
#         elif 1 == self.input_shape[2]:
#             color_mode = 'grayscale'

#         train_ds = tf.keras.utils.image_dataset_from_directory(
#             directory= self.path_to_training_set,
#             labels='inferred',
#             label_mode='categorical',
#             color_mode=color_mode,
#             batch_size=self.batch_size,
#             image_size=self.input_shape[0:2],
#             shuffle=True,
#             seed=FIXED_SEED,
#             validation_split=self.val_split,
#             subset='training'
#         )

#         validation_ds = tf.keras.utils.image_dataset_from_directory(
#             directory= self.path_to_training_set,
#             labels='inferred',
#             label_mode='categorical',
#             color_mode=color_mode,
#             batch_size=self.batch_size,
#             image_size=self.input_shape[0:2],
#             shuffle=True,
#             seed=FIXED_SEED,
#             validation_split=self.val_split,
#             subset='validation'
#         )


#         if self.cache :
#             self.train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
#             self.validation_ds = validation_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
#         else :
#             self.train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
#             self.validation_ds = validation_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

#     def quantize_model_uint8(self):
#         def representative_dataset():
#             for data in self.train_ds.rebatch(1).take(150):
#                 yield [tf.dtypes.cast(data[0], tf.float32)]

#         model = tf.keras.models.load_model(self.path_to_trained_models / f"{self.model_name}.h5")
#         converter = tf.lite.TFLiteConverter.from_keras_model(model)
#         converter.optimizations = [tf.lite.Optimize.DEFAULT]
#         converter.representative_dataset = representative_dataset
#         converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
#         converter.inference_input_type = tf.uint8
#         converter.inference_output_type = tf.uint8
#         tflite_quant_model = converter.convert()

#         with open(self.path_to_trained_models / f"{self.model_name}.tflite", "wb") as f:
#             f.write(tflite_quant_model)

#         (self.path_to_trained_models / f"{self.model_name}.h5").unlink()

#     def evaluate_flash_and_peak_RAM_occupancy(self):
#         # quantize model to evaluate peak RAM and Flash occupancy
#         self.quantize_model_uint8()

#         # evaluate peak RAM and Flash occupancy using STMicroelectronics' X-CUBE-AI
#         proc = subprocess.Popen(
#             [self.path_to_stm32tflm, self.path_to_trained_models / f"{self.model_name}.tflite"], 
#             stdout=subprocess.PIPE
#         )
#         try:
#             outs, errs = proc.communicate(timeout=15)
#             Flash, RAM = re.findall(r'\d+', str(outs))
#         except subprocess.TimeoutExpired:
#             proc.kill()
#             outs, errs = proc.communicate()
#             print("stm32tflm error")
#             exit()

#         return int(Flash), int(RAM)

#     def evaluate_model_process(self, k, c):
#         if k > 0:
#             self.model_name = f"k_{k}_c_{c}"
#             print(f"\n{self.model_name}\n")
#             checkpoint = tf.keras.callbacks.ModelCheckpoint(
#                 str(self.path_to_trained_models / f"{self.model_name}.h5"), monitor='val_accuracy',
#                 verbose=0, save_best_only=True, save_weights_only=False, mode='auto'
#             )
#             # [Fix Randomness] Reset the global seed for each NAS iteration to ensure every new model architecture starts with the exact same initial weight sequence
#             #tf.keras.backend.clear_session()
#             #tf.keras.utils.set_random_seed(FIXED_SEED)
#             model, MACC, number_of_cells_limited = self.Model(k, c)
#             # One epoch of training must be done before quantization 
#             # which is needed to evaluate RAM and Flash occupancy
#             model.fit(self.train_ds, epochs=1, validation_data=self.validation_ds, validation_freq=1, verbose=0)
#             model.save(self.path_to_trained_models / f"{self.model_name}.h5")
#             Flash, RAM = self.evaluate_flash_and_peak_RAM_occupancy()
#             print(f"\nRAM: {RAM},\tFlash: {Flash},\tMACC: {MACC}\n")
            
#             # If sastisfy hardware-constraints
#             # RAM, Flash, MACC, maximum cells
#             if MACC <= self.max_MACC and RAM <= self.max_RAM and Flash <= self.max_Flash and not number_of_cells_limited:
#                 hist = model.fit(self.train_ds, epochs=self.epochs - 1, validation_data=self.validation_ds, validation_freq=1, callbacks=[checkpoint], verbose=0)
#                 self.quantize_model_uint8()
#                 print(hist.history['val_accuracy'])
    
#             return {
#                 'k': k,
#                 'c': c if not number_of_cells_limited else "Not feasible",
#                 'RAM': RAM if RAM <= self.max_RAM else "Outside the upper bound",
#                 'Flash': Flash if Flash <= self.max_Flash else "Outside the upper bound",
#                 'MACC': MACC if MACC <= self.max_MACC else "Outside the upper bound",
#                 'max_val_acc': np.around(np.amax(hist.history['val_accuracy']), decimals=3)
#                 if 'hist' in locals() else -3
#             }
                    
#         else:
#             return {
#                 'k': 'unfeasible',
#                 'c': c,
#                 'max_val_acc' : -3
#             }
    
#     # the original version
#     def explore_num_cells(self, k):
#         previous_architecture = {'k': -1, 'c': -1, 'max_val_acc': -2}
#         current_architecture = {'k': -1, 'c': -1, 'max_val_acc': -1}
#         c = -1
#         k = int(k)
    
#         while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc']):
#             previous_architecture = current_architecture
#             c += 1
#             self.model_counter += 1
#             current_architecture = self.evaluate_model_process(k, c)
#             print(f"\n\n\n{current_architecture}\n\n\n")
            
#         return previous_architecture
# #----------------------------------------------------------------------------------------------
#     # uncomment to use the "trying more cells" version
#     #def explore_num_cells(self, k):
#     #    k = int(k)
#     #    c = 0
#     #    best_c = -1
#     #    best_accuracy = -1
#     #    max_attempts = 2
#     #    
#     #    best_architecture = {'k': -1, 'c': -1, 'max_val_acc': -1}  
#     #    worse = 0
#     #    current_architecture = self.evaluate_model_process(k, c)
#     #    self.model_counter += 1
#     #    
#     #    print(f"\n\n\n{current_architecture}\n\n\n")
#     #    
#     #    while current_architecture['max_val_acc'] != -3:
#     #        if current_architecture['max_val_acc'] > best_accuracy:
#     #            best_accuracy = current_architecture['max_val_acc']
#     #            best_c = c
#     #            best_architecture = current_architecture 
#     #            worse = 0  
#     #        else:
#     #            worse += 1
#     #            
#     #        if worse >= max_attempts:
#     #            break
#     #        c += 1
#     #        current_architecture = self.evaluate_model_process(k, c)
#     #        self.model_counter += 1
#     #        
#     #        print(f"\n\n\n{current_architecture}\n\n\n")
#     #        
#     #    return best_architecture
# #-----------------------------------------------------------------------------------------
#     # uncomment to use the backward traversal version
#     #def explore_num_cells(self, k):
#     #    k = int(k)

#     #    original_epochs = self.epochs
#     #    self.epochs = 2 
#     #    
#     #    c = 0
#     #    best_architecture = {'k': -1, 'c': -1, 'max_val_acc': -3}
        
#     #    current_eval = self.evaluate_model_process(k, c)
#     #    self.model_counter += 1
#     #    print(f"\n\n\n{current_eval}\n\n\n")

#     #    if current_eval['max_val_acc'] == -3:
#     #        self.epochs = original_epochs
#     #        return best_architecture
#     #    
#     #    while current_eval['max_val_acc'] != -3:
#     #        c += 1
#     #        current_eval = self.evaluate_model_process(k, c)
#     #        self.model_counter += 1
#     #        print(f"\n\n\n{current_eval}\n\n\n")
#     #        
#     #    max_c = c - 1

#     #    self.epochs = original_epochs
#     #    
#     #    c = max_c
#     #    best_accuracy = -3

#     #    while c >= 0:
#     #        current_architecture = self.evaluate_model_process(k, c)
#     #        self.model_counter += 1
#     #        
#     #        print(f"\n{current_architecture}\n")
#     #        
#     #        if current_architecture['max_val_acc'] >= best_accuracy:
#     #            best_accuracy = current_architecture['max_val_acc']
#     #            best_architecture = current_architecture
#     #            c -= 1
#     #        else:
#     #            break

#     #    return best_architecture

#     def search(self):
#         self.model_counter = 0
#         epsilon = 0.005
#         k0 = 4

#         start = datetime.datetime.now()

#         k = k0
#         previous_architecture = self.explore_num_cells(k)
#         k = 2 * k
#         current_architecture = self.explore_num_cells(k)

#         if current_architecture['max_val_acc'] > previous_architecture['max_val_acc']:
#             previous_architecture = current_architecture
#             k *= 2
#             current_architecture = self.explore_num_cells(k)
            
#             while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc'] + epsilon):
#                 previous_architecture = current_architecture
#                 k *= 2
#                 current_architecture = self.explore_num_cells(k)
                
#         else:
#             k = k0 / 2
#             current_architecture = self.explore_num_cells(k)

#             while(current_architecture['max_val_acc'] >= previous_architecture['max_val_acc']):
#                 previous_architecture = current_architecture
#                 k /= 2
#                 current_architecture = self.explore_num_cells(k)

#         resulting_architecture = previous_architecture

#         end = datetime.datetime.now()

#         if resulting_architecture['max_val_acc'] > 0:
#             resulting_architecture_name = f"k_{resulting_architecture['k']}_c_{resulting_architecture['c']}.tflite"
#             self.path_to_resulting_architecture = self.save_path / f"resulting_architecture_{resulting_architecture_name}"
#             (self.path_to_trained_models / f"{resulting_architecture_name}").rename(self.path_to_resulting_architecture)
#             shutil.rmtree(self.path_to_trained_models)
#             print(f"\nResulting architecture: {resulting_architecture}\n")
            
#         else:
#             print(f"\nNo feasible architecture found\n")
            
#         print(f"Elapsed time (search): {end-start}\n")

#         return self.path_to_resulting_architecture

In [ ]:
# Improve vers
class ColabNAS :
    def __init__(self, max_RAM, max_Flash, max_MACC, path_to_training_set, 
                 val_split, cache=False, input_shape=(50,50,3), save_path='.',
                 path_to_stm32tflm='/kaggle/working/stm32tflm'):        
        self.learning_rate = 1e-3
        self.batch_size = 128
        self.epochs = 100 

        self.max_MACC = max_MACC
        self.max_Flash = max_Flash
        self.max_RAM = max_RAM
        self.path_to_training_set = path_to_training_set
        self.num_classes = len(next(os.walk(path_to_training_set))[1])
        self.val_split = val_split
        self.cache = cache
        self.input_shape = input_shape
        self.save_path = Path(save_path)

        self.path_to_trained_models = self.save_path / "trained_models"
        self.path_to_trained_models.mkdir(parents=True, exist_ok=True)

        self.path_to_stm32tflm = Path(path_to_stm32tflm)

        self.load_training_set()

    # k: number of kernels of the first convolutional layer
    # c: number of cells added upon the first convolutional layer
    # pre-processing pipeline not included in MACC computation
    def Model(self, k, c):
        # [Fix Randomness] Fix random seed for weight initialization to guarantee consistent model convergence
        #init = tf.keras.initializers.GlorotUniform(seed=FIXED_SEED)
        kernel_size = (3, 3)
        pool_size = (2, 2)
        pool_strides = (2, 2)

        number_of_cells_limited = False
        number_of_mac = 0

        # inputs = (50, 50, 3) by default
        inputs = keras.Input(shape=self.input_shape)

        # preprocessing pipeline
        x = tf.keras.layers.RandomFlip('horizontal')(inputs)
        x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
        x = tf.keras.layers.Rescaling(1./255)(x)
        x = tf.keras.layers.BatchNormalization()(x)

        # convolutional base
        n = k
        multiplier = 2

        # first convolutional layer
        # c_in = 3 now
        c_in = self.input_shape[2]
        # x.shape() = (batch_size, 50, 50, n)
        x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
        # MAC = 3 * (3x3) * (50x50) * n
        number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        # adding cells
        for i in range(1, c + 1):
            if x.shape[1] <= 1 or x.shape[2] <= 1:
                number_of_cells_limited = True
                break
            n = int(np.ceil(n * multiplier))
            multiplier = multiplier - 2**-i
            # x.shape() = (batch_size, h_old /2, w_old /2, n_old)
            x = keras.layers.MaxPooling2D(pool_size=pool_size, strides=pool_strides, padding='valid')(x)
            # c_in = n_old
            c_in = x.shape[3]
            # x.shape() = (batch_size, h, w, n_new)
            x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
            # MAC = c_in * (3x3) * (hxw) * n_new
            number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        # classifier
        # x.shape() = (batch_size, n_last)
        x = keras.layers.GlobalAveragePooling2D()(x)
        # input_shape = n_last
        input_shape = x.shape[1]
        # x.shape() = (batch_size, n_last)
        x = keras.layers.Dense(n, activation='relu')(x)
        number_of_mac += (input_shape * x.shape[1])
        # outputs.shape() = (batch_size, num_classes)
        outputs = keras.layers.Dense(self.num_classes, activation='softmax')(x)
        number_of_mac += (x.shape[1] * outputs.shape[1])

        model = keras.Model(inputs=inputs, outputs=outputs)

        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        model.compile(optimizer=optimizer,
                loss='categorical_crossentropy',
                metrics=['accuracy'])
        
        model.summary()

        return model, number_of_mac, number_of_cells_limited

    def load_training_set(self):
        if 3 == self.input_shape[2]:
            color_mode = 'rgb'
        elif 1 == self.input_shape[2]:
            color_mode = 'grayscale'

        train_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=FIXED_SEED,
            validation_split=self.val_split,
            subset='training'
        )

        validation_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=FIXED_SEED,
            validation_split=self.val_split,
            subset='validation'
        )


        if self.cache :
            self.train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
            self.validation_ds = validation_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
        else :
            self.train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
            self.validation_ds = validation_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

    def quantize_model_uint8(self):
        def representative_dataset():
            for data in self.train_ds.rebatch(1).take(150):
                yield [tf.dtypes.cast(data[0], tf.float32)]

        model = tf.keras.models.load_model(self.path_to_trained_models / f"{self.model_name}.h5")
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
        tflite_quant_model = converter.convert()

        with open(self.path_to_trained_models / f"{self.model_name}.tflite", "wb") as f:
            f.write(tflite_quant_model)

        (self.path_to_trained_models / f"{self.model_name}.h5").unlink()

    def evaluate_flash_and_peak_RAM_occupancy(self):
        # quantize model to evaluate peak RAM and Flash occupancy
        self.quantize_model_uint8()

        # evaluate peak RAM and Flash occupancy using STMicroelectronics' X-CUBE-AI
        proc = subprocess.Popen(
            [self.path_to_stm32tflm, self.path_to_trained_models / f"{self.model_name}.tflite"], 
            stdout=subprocess.PIPE
        )
        try:
            outs, errs = proc.communicate(timeout=15)
            Flash, RAM = re.findall(r'\d+', str(outs))
        except subprocess.TimeoutExpired:
            proc.kill()
            outs, errs = proc.communicate()
            print("stm32tflm error")
            exit()

        return int(Flash), int(RAM)

    def evaluate_model_process(self, k, c):
        if k > 0:
            self.model_name = f"k_{k}_c_{c}"
            print(f"\n{self.model_name}\n")
            checkpoint = tf.keras.callbacks.ModelCheckpoint(
                str(self.path_to_trained_models / f"{self.model_name}.h5"), monitor='val_accuracy',
                verbose=0, save_best_only=True, save_weights_only=False, mode='auto'
            )
            
            # [Fix Randomness] Reset the global seed for each NAS iteration to ensure every new model architecture starts with the exact same initial weight sequence
            #tf.keras.backend.clear_session()
            #tf.keras.utils.set_random_seed(FIXED_SEED)
            model, MACC, number_of_cells_limited = self.Model(k, c)
            # One epoch of training must be done before quantization 
            # which is needed to evaluate RAM and Flash occupancy
            model.fit(
                self.train_ds, 
                epochs=1, 
                validation_data=self.validation_ds, 
                validation_freq=1, 
                verbose=0
            )
            
            model.save(self.path_to_trained_models / f"{self.model_name}.h5")
            Flash, RAM = self.evaluate_flash_and_peak_RAM_occupancy()
            print(f"\nRAM: {RAM},\tFlash: {Flash},\tMACC: {MACC}\n")
            
            # If sastisfy hardware-constraints
            # RAM, Flash, MACC, maximum cells
            if MACC <= self.max_MACC and RAM <= self.max_RAM and Flash <= self.max_Flash and not number_of_cells_limited:
                
                hist = model.fit(
                    self.train_ds, 
                    epochs=self.epochs - 1, 
                    validation_data=self.validation_ds, 
                    validation_freq=1, 
                    callbacks=[checkpoint], 
                    verbose=0
                )
                
                self.quantize_model_uint8()
                print(hist.history['val_accuracy'])
    
            return {
                'k': k,
                'c': c if not number_of_cells_limited else "Not feasible",
                'RAM': RAM if RAM <= self.max_RAM else "Outside the upper bound",
                'Flash': Flash if Flash <= self.max_Flash else "Outside the upper bound",
                'MACC': MACC if MACC <= self.max_MACC else "Outside the upper bound",
                'max_val_acc': np.around(np.amax(hist.history['val_accuracy']), decimals=3)
                if 'hist' in locals() else -3
            }
                    
        else:
            return {
                'k': 'unfeasible',
                'c': c,
                'max_val_acc' : -3
            }
    
    # the original version
    def explore_num_cells(self, k):
        previous_architecture = {'k': -1, 'c': -1, 'max_val_acc': -2}
        current_architecture = {'k': -1, 'c': -1, 'max_val_acc': -1}
        c = -1
        k = int(k)
    
        while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc']):
            previous_architecture = current_architecture
            c += 1
            self.model_counter += 1
            current_architecture = self.evaluate_model_process(k, c)
            print(f"\n\n\n{current_architecture}\n\n\n")
            
        return previous_architecture

    def search(self):
        self.model_counter = 0
        epsilon = 0.005
        k0 = 4

        start = datetime.datetime.now()

        k = k0
        previous_architecture = self.explore_num_cells(k)
        k = 2 * k
        current_architecture = self.explore_num_cells(k)

        if current_architecture['max_val_acc'] > previous_architecture['max_val_acc']:
            previous_architecture = current_architecture
            k *= 2
            current_architecture = self.explore_num_cells(k)
            
            while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc'] + epsilon):
                previous_architecture = current_architecture
                k *= 2
                current_architecture = self.explore_num_cells(k)
                
        else:
            k = k0 / 2
            current_architecture = self.explore_num_cells(k)

            while(current_architecture['max_val_acc'] >= previous_architecture['max_val_acc']):
                previous_architecture = current_architecture
                k /= 2
                current_architecture = self.explore_num_cells(k)

        resulting_architecture = previous_architecture

        end = datetime.datetime.now()

        if resulting_architecture['max_val_acc'] > 0:
            resulting_architecture_name = f"k_{resulting_architecture['k']}_c_{resulting_architecture['c']}.tflite"
            self.path_to_resulting_architecture = self.save_path / f"resulting_architecture_{resulting_architecture_name}"
            (self.path_to_trained_models / f"{resulting_architecture_name}").rename(self.path_to_resulting_architecture)
            shutil.rmtree(self.path_to_trained_models)
            print(f"\nResulting architecture: {resulting_architecture}\n")
            
        else:
            print(f"\nNo feasible architecture found\n")
            
        print(f"Elapsed time (search): {end-start}\n")

        return self.path_to_resulting_architecture

Hàm `test`

In [7]:
def test_tflite_model(path_to_resulting_model, test_ds):
    interpreter = tf.lite.Interpreter(str(path_to_resulting_model))
    interpreter.allocate_tensors()

    output = interpreter.get_output_details()[0]
    input = interpreter.get_input_details()[0]

    correct = 0
    wrong = 0

    for i in test_ds:
        image, label = i[0], i[1]
        # Check if input_type is quantized, then rescale input data to uint8
        if input['dtype'] == tf.uint8:
            input_scale, input_zero_point = input['quantization']
            image = image / input_scale + input_zero_point
        input_data = tf.dtypes.cast(image, tf.uint8)
        interpreter.set_tensor(input['index'], input_data)
        interpreter.invoke()

        if label.numpy().argmax() == interpreter.get_tensor(output['index']).argmax():
            correct += 1
        else:
            wrong += 1

    print(f"\nTflite model test accuracy: {correct/(correct+wrong)}")

# Thí nghiệm 1 - Chạy thí nghiệm giống bài báo gốc

## 1. Các bộ dữ liệu

### 1.1 Melanoma Skin Cancer

Bài toán phân loại Ung thư Da hắc tố (**Melanoma Skin Cancer**) nhằm mục đích phân biệt giữa hình ảnh lành tính (**benign**) và ác tính (**malignant**) của ung thư da hắc tố. Nó bao gồm một tập train chứa **9605** ảnh và một tập kiểm tra gồm **1000** ảnh.

### 1.2 Tập dữ liệu Flowers-4

Bài toán phân loại trên tập **Flowers-4** (là tập con của tập dữ liệu **Flowers**) nhằm mục đích phân biệt giữa bốn loại hoa: bồ công anh (**dandelion**), diên vĩ (**iris**), tulip (**tulip**) và mộc lan (**magnolia**). Nó chứa **1052** mẫu bồ công anh, **1054** mẫu diên vĩ, **1048** mẫu tulip và **1048** mẫu mộc lan.

### 1.3 Tập dữ liệu Animals-3

Bài toán phân loại trên tập dữ liệu **Animals-3** (là tập con của tập dữ liệu **Animals-10**) nhằm mục đích phân biệt giữa ba loài động vật: ngựa (**horse**), bướm (**butterfly**) và gà (**chicken**). Nó chứa **2623** trường hợp ngựa, **2112** trường hợp bướm và **3098** trường hợp gà.

### 1.4 Tập dữ liệu MNIST

### 1.5 Tập dữ liệu Visual Wake Words

## 2. Thí nghiệm hardware-aware trên các phần cứng hạn chế

### 2.1 Cấu hình thí nghiệm

In [8]:
hw_batch_size = 32

hw_input_shape = (50, 50, 3)

# Dataset directory
data_dirs = {
    'melanoma' : Path("/kaggle/input/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"),
    'flowers' : Path("/kaggle/input/datasets/karosvn/flowers-4/flowers"),
    'animals' : Path("/kaggle/input/datasets/karosvn/animals-3/animals"),
    'mnist' : Path("/kaggle/input/datasets/tommerfrancis/mnist-dataset/MNIST"),
    'vww' : Path("/kaggle/input/datasets/tommerfrancis/visual-wake-words/visual_wake_words")
}

# Target: STM32L010RBT6, STM32L151UCY6DTR, STM32L412KBU3
hardware_configs = {
    'L0' : {
        'RAM': 20480,
        'Flash': 131072,
        'MACC': 750000 # CoreMark * 10e4        
    },
    'L1': {
        'RAM': 32768,
        'Flash': 262144,
        'MACC': 930000 # CoreMark * 10e4
    },
    'L4': {
        'RAM': 40960,
        'Flash': 131072,
        'MACC': 2730000 # CoreMark * 10e4
    }
}

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
hw_val_split = 0.3

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
hw_cache = True

# where to save results
hw_save_path = '/kaggle/working/'

### 2.2 Hàm helper cho thí nghiệm chạy `ColabNAS` trên các cấu hình phần cứng hạn chế

Hàm khởi tạo class `ColabNAS` và thực hiện thuật toán trên phần cứng `target` cụ thể

In [9]:
def searchOnHardware(target, data_dir, max_RAM, max_Flash, max_MACC,
                     val_split, cache, input_shape=(50, 50, 3), save_path='.'):

    print(f"\n\n\n======\t\tRun on target {target}\t\t==============\n\n\n")
    colabNAS = ColabNAS(
        max_RAM=max_RAM,
        max_Flash=max_Flash,
        max_MACC=max_MACC,
        path_to_training_set=data_dir / "train",
        val_split=val_split,
        cache=cache,
        input_shape=input_shape,
        save_path=save_path
    )

    # Search
    path_to_resulting_model = colabNAS.search()
    return path_to_resulting_model

Hàm thực hiện đánh giá mô hình trên tập `test`

In [10]:
def testModel(data_dir, path_to_resulting_model):

    test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "test",
        image_size=hw_input_shape[0:2],
        batch_size=hw_batch_size,
        shuffle=False
    )

    # One-hot for label
    class_names = test_ds.class_names
    num_classes = len(class_names)

    test_ds  = test_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)))
    test_ds = test_ds.unbatch().batch(1)

    for target, model_path in path_to_resulting_model.items():
        print(f"\n===== Testing on {target} =====")
        test_tflite_model(model_path, test_ds)


### 2.3 Chạy thí nghiệm trên từng bộ dữ liệu

Chọn tập dữ liệu từ `data_dirs`

In [11]:
# data_dir = data_dirs['melanoma'] # Tập Melanoma
# hoặc
data_dir = data_dirs['flowers'] # Tập Flowers-4
# data_dir = data_dirs['animals'] # Tập Animals-3
# data_dir = data_dirs['mnist']   # Tập MNIST
# data_dir = data_dirs['vww']     # Tập Visual Wake Words

Chạy thuật toán `ColabNAS` trên các phần cứng trong `hardware_configs`

In [12]:
path_to_resulting_model = {}

for target in hardware_configs:
    path_to_resulting_model[target] = searchOnHardware(
        target=target,
        data_dir=data_dir,
        max_RAM=hardware_configs[target]['RAM'],
        max_Flash=hardware_configs[target]['Flash'],
        max_MACC=hardware_configs[target]['MACC'],
        val_split=hw_val_split,
        cache=hw_cache,
        save_path=hw_save_path
    )




======		Run on target L0		==============



Found 3360 files belonging to 4 classes.
Using 2352 files for training.


I0000 00:00:1784817554.424641      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784817554.430678      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3360 files belonging to 4 classes.
Using 1008 files for validation.

k_4_c_0



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 50, 50, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 50, 3)      │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 50, 4)      │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 4)              │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │            20 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            20 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

INFO:tensorflow:Assets written to: /tmp/tmpd5glqoln/assets


Saved artifact at '/tmp/tmpd5glqoln'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 3), dtype=tf.float32, name='input_layer_34')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  135854987026576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987019664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987018320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987019280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987021200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987019472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987020240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987027152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987026192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854987019856: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1784818267.769940      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1784818267.769963      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.



RAM: 50688,	Flash: 6184,	MACC: 1080320




{'k': 16, 'c': 0, 'RAM': 'Outside the upper bound', 'Flash': 6184, 'MACC': 1080320, 'max_val_acc': -3}




Resulting architecture: {'k': 8, 'c': 5, 'RAM': 34304, 'Flash': 47056, 'MACC': 2086403, 'max_val_acc': np.float64(0.916)}

Elapsed time (search): 0:05:08.880915



fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Test các mô hình kết quả

In [13]:
testModel(data_dir, path_to_resulting_model)

Found 842 files belonging to 4 classes.

===== Testing on L0 =====


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



Tflite model test accuracy: 0.7600950118764845

===== Testing on L1 =====

Tflite model test accuracy: 0.8729216152019003

===== Testing on L4 =====

Tflite model test accuracy: 0.8966745843230404


## 3. Chạy Transfer Learning

### 3.1 Cấu hình thí nghiệm

In [14]:
tl_input_shape = (224, 224, 3)

# fine-tuned MobileNetV2's params (in byte)
tl_max_RAM = 2583552
tl_max_Flash = 2713592
tl_max_MACC = 300000000 # https://arxiv.org/pdf/1801.04381.pdf

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
tl_val_split = 0.3

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
tl_cache = True

# where to save results
tl_save_path = '/kaggle/working/'

tl_batch_size = 128
epochs_transfer_learning = 20
epochs_fine_tuning = 10
epochs_colabnas = 100

### 3.2 Hàm helper chạy thí nghiệm so sánh với Transfer Learning

In [21]:
# load and preprocess image dataset for transfer learning model training and evaluation
def load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape):
    num_classes = len(next(os.walk(path_to_training_set))[1])
    
    train_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_training_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = True,
        seed = FIXED_SEED,
        validation_split = validation_split,
        subset = "training"
    )

    validation_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_training_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = True,
        seed = FIXED_SEED,
        validation_split = validation_split,
        subset = "validation"
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_test_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = False,
        #seed = 11
    )

    # cache train and validation sets in memory (RAM)
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
    validation_ds = validation_ds.cache().prefetch(buffer_size = AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size = AUTOTUNE)

    return train_ds, validation_ds, test_ds, num_classes

def quantize_model_uint8(train_ds, model_name): # apply post training quantization to transfer learning model
    def representative_dataset(): # provide small size of samples for calibration to determine activation ranges for accurate int8 quantization
        for data in train_ds.rebatch(1).take(150):
            yield [tf.dtypes.cast(data[0], tf.float32)]

    model = tf.keras.models.load_model(f"{model_name}.h5")
    converter = tf.lite.TFLiteConverter.from_keras_model(model) # convert model to TensorFlow Lite
    converter.optimizations = [tf.lite.Optimize.DEFAULT] # enable int8 quantization
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] # force strictly int8 operations
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_quant_model = converter.convert()

    with open(f"{model_name}.tflite", "wb") as f:
        f.write(tflite_quant_model)
    os.remove(f"{model_name}.h5")


Class `TLModel`

In [25]:
class TLModel:
    def __init__(self, initializer, input_shape, num_classes, 
                 epochs_train, epochs_finetune, save_path='.'):
        
        self.base_model = tf.keras.applications.MobileNetV2( # MobileNetV2 is used as backbone
            weights = "imagenet", # load weights pre-trained on ImageNet
            input_shape = tl_input_shape, 
            include_top = False) # drop original 1000-class classification layer of MobileNetV2
        
        self.base_model.trainable = False # freeze CNN layers

        self.initializer = initializer
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.epochs_train = epochs_train
        self.epochs_finetune = epochs_finetune
        self.save_path = save_path

        # auto-save model with highest validation accuracy during training
        self.checkpoint = tf.keras.callbacks.ModelCheckpoint(
            save_path + ".h5", 
            monitor="val_accuracy", 
            verbose=0,
            save_best_only=True, 
            save_weights_only=False, 
            mode="auto"
        )
        

    def pipeline(self):

        inputs = tf.keras.Input(shape=self.input_shape)

        # pre-processing pipeline
        x = tf.keras.layers.RandomFlip("horizontal")(inputs) # horizontal flips
        x = tf.keras.layers.RandomRotation(factor = 0.2, fill_mode = "constant", interpolation = "bilinear")(x)
        x = tf.keras.layers.Rescaling(1/255)(x) # min-max standardization
        x = tf.keras.layers.BatchNormalization()(x)

        # The base model contains batchnorm layers. 
        # We want to keep them in inference mode when we unfreeze the base model for fine-tuning,
        # so we make sure that the base model is running in inference mode here.
        x = self.base_model(x, training=False)
        
        # custom classifier
        x = tf.keras.layers.GlobalAveragePooling2D()(x)

        # single fully connected layer
        outputs = tf.keras.layers.Dense(self.num_classes, activation = "softmax", kernel_initializer=self.initializer)(x) 
        model = tf.keras.Model(inputs=inputs, outputs=outputs)

        # optimizer and compile
        optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3)
        model.compile(
            optimizer=optimizer, 
            loss="categorical_crossentropy",
            metrics=["accuracy"])
        
        self.model = model
        model.summary()

    def train_classifier(self, train_ds, val_ds):
        # train custom classifier
        self.model.fit(
            x=train_ds, 
            epochs=self.epochs_train, 
            validation_data=val_ds, 
            verbose=0,
            validation_freq=1)
        
        self.base_model.trainable = True # unfreeze CNN layers to fine-tune model

    def finetune(self, train_ds, val_ds):
        # optimizer and compile
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5) 
        self.model.compile(
            optimizer=optimizer, 
            loss="categorical_crossentropy", 
            metrics=["accuracy"])

        # fine-tune model
        self.model.fit(
            x=train_ds, 
            epochs=self.epochs_finetune,
            validation_data=val_ds, 
            callbacks = [self.checkpoint], 
            verbose=0, 
            validation_freq = 1)
        
    def evaluate(self, test_ds):
        self.model.load_weights(self.save_path + ".h5") # load best weights
        test_acc = self.model.evaluate(test_ds, return_dict=True) # evaluate model on test dataset
        
        return test_acc


### 3.2 Chạy thí nghiệm

Chọn tập dữ liệu từ `data_dirs`

In [17]:
# data_dir = data_dirs['melanoma'] # Tập Melanoma
# hoặc
data_dir = data_dirs['flowers'] # Tập Flowers-4
# data_dir = data_dirs['animals'] # Tập Animals-3

#### Transfer Learning

In [26]:
# [Fix Randomness] Clear the previous Keras session to provide a clean state before initializing the random seed
tf.keras.backend.clear_session()
# [Fix Randomness] Set global seed for reproducible results across Python, NumPy, and Keras
tf.keras.utils.set_random_seed(FIXED_SEED)
# [Fix Randomness] Enable deterministic operations to prevent GPU floating-point variations
tf.config.experimental.enable_op_determinism()
# [Fix Randomness] Fix random seed for weight initialization to guarantee consistent model convergence
initializer = tf.keras.initializers.GlorotUniform(seed=FIXED_SEED)

# Load dataset
train_ds, val_ds, test_ds, num_classes = load_dataset(
    path_to_training_set=data_dir / "train", 
    path_to_test_set=data_dir / "test", 
    batch_size=tl_batch_size, 
    validation_split=tl_val_split, 
    input_shape=tl_input_shape)

# START TIMER
start = datetime.datetime.now() 

# init Transfer Learning model
tl_model = TLModel(
    initializer=initializer,
    input_shape=tl_input_shape,
    num_classes=num_classes,
    epochs_train=epochs_transfer_learning,
    epochs_finetune=epochs_fine_tuning,
    save_path=tl_save_path
)

# create model pipeline
tl_model.pipeline()

# train classifier
tl_model.train_classifier(train_ds, val_ds)

# fine-tune model
tl_model.finetune(train_ds, val_ds)

# STOP TIMER
end = datetime.datetime.now() 
print(f"\nTraining time: {end - start}\n") 

Found 3360 files belonging to 4 classes.
Using 2352 files for training.
Found 3360 files belonging to 4 classes.
Using 1008 files for validation.
Found 842 files belonging to 4 classes.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 3)    │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,120 (8.63 MB)

 Trainable params: 5,130 (20.04 KB)

 Non-trainable params: 2,257,990 (8.61 MB)


Training time: 0:06:59.808084



#### ColabNAS

In [27]:
# initialize ColabNAS with MobileNetV2 constraints
colabNAS = ColabNAS(
    max_RAM=tl_max_RAM, 
    max_Flash=tl_max_Flash, 
    max_MACC=tl_max_MACC,
    path_to_training_set=data_dir / "train", 
    val_split=tl_val_split, 
    cache=tl_cache, 
    input_shape=tl_input_shape, 
    save_path=tl_save_path)

# search
path_to_tflite_model = colabNAS.search()

Found 3360 files belonging to 4 classes.
Using 2352 files for training.
Found 3360 files belonging to 4 classes.
Using 1008 files for validation.

k_4_c_0



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 3)    │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 4)    │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 4)              │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            20 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            20 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164 (656.00 B)

 Trainable params: 158 (632.00 B)

...

INFO:tensorflow:Assets written to: /tmp/tmp4gswja_f/assets


Saved artifact at '/tmp/tmp4gswja_f'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  135854547576272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547576656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547576080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547575696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547568208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547575312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547574544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547576848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547577808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135854547578000: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1784820009.828839      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1784820009.828884      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8



RAM: 354304,	Flash: 4848,	MACC: 5419040



KeyboardInterrupt: 

Test ColabNAS and Transfer Learning

In [28]:
# Test TL model
print("\n========Transfer Learning========\n")
test_acc = tl_model.evaluate(test_ds)
print(f"Test Accuracy: \n{test_acc}")

# Test ColabNAS
print("\n========ColabNAS========\n")
test_ds = tf.keras.utils.image_dataset_from_directory(
    directory = data_dir / "test",
    labels = "inferred",
    label_mode = "categorical",
    color_mode = "rgb",
    batch_size = tl_batch_size,
    image_size = tl_input_shape[0:2],
    shuffle = False,
    #seed = 
)

test_ds = test_ds.unbatch().batch(1)
test_tflite_model(path_to_tflite_model, test_ds) # evaluate optimal model on test dataset


========Transfer Learning========

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - accuracy: 0.9809 - loss: 0.0540
Test Accuracy: 
{'accuracy': 0.978622317314148, 'loss': 0.06165621429681778}

========ColabNAS========

Found 842 files belonging to 4 classes.


NameError: name 'path_to_tflite_model' is not defined

## 4. Chạy thí nghiệm so sánh ColabNAS với SOTA trên tập VWW

Chọn tập dữ liệu từ `data_dirs`

In [ ]:
data_dir = data_dirs['vww'] 

### 4.1 Cấu hình thí nghiệm

In [ ]:
sota_input_shape = (50, 50, 3)

# target: STMF446RE
sota_max_RAM = 131072
sota_max_Flash = 524288
sota_max_MACC = 6080000 # CoreMark * 10^4

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
sota_val_split = 0.3
sota_batch_size = 32

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
sota_cache = True

# where to save results
sota_save_path = '/kaggle/working/'

### 4.2 ColabNAS

In [ ]:
# initialize ColabNAS with STMF446RE constraints
colabNAS = ColabNAS(
    max_RAM=sota_max_RAM, 
    max_Flash=sota_max_Flash, 
    max_MACC=sota_max_MACC,
    path_to_training_set=data_dir / "train", 
    val_split=sota_val_split, 
    cache=sota_cache, 
    input_shape=sota_input_shape, 
    save_path=sota_save_path)

# search
path_to_tflite_model = colabNAS.search()

Test ColabNAS

In [ ]:
# Test ColabNAS
print("\n========ColabNAS========\n")
test_ds = tf.keras.utils.image_dataset_from_directory(
    directory = data_dir / "test",
    labels = "inferred",
    label_mode = "categorical",
    color_mode = "rgb",
    batch_size = sota_batch_size,
    image_size = sota_input_shape[0:2],
    shuffle = False,
    #seed = 
)

test_ds = test_ds.unbatch().batch(1)
test_tflite_model(path_to_tflite_model, test_ds) # evaluate optimal model on test dataset

In [ ]:
!/kaggle/working/stm32tflm $path_to_tflite_model # run TFLite model on STM32TFLM simulator

In [ ]:
interpreter = tf.lite.Interpreter(str(path_to_tflite_model)) # load optimal ColabNAS model into TFLite interpreter for evaluation
interpreter.allocate_tensors() # allocate memory for model's tensors before evaluation
%timeit interpreter.invoke() # measure average execution time of optimal ColabNAS model